In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_F.xpt"
csv_file = DATA_DIR / "WHQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 WHQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD045",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080L",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHD080T",
    "WHQ270",
    "WHQ280A",
    "WHQ280B",
    "WHQ280C",
    "WHQ280D",
    "WHQ280E",
    "WHQ090",
    "WHD100A",
    "WHD100B",
    "WHD100C",
    "WHD100D",
    "WHD100E",
    "WHD100F",
    "WHD100G",
    "WHD100H",
    "WHD100I",
    "WHD100J",
    "WHD100K",
    "WHD100L",
    "WHD100M",
    "WHD100N",
    "WHD100O",
    "WHD100P",
    "WHD100Q",
    "WHD100R",
    "WHD100S",
    "WHD100T",
    "WHQ210",
    "WHD220",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "WHQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 6889:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 63:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,889 rows x 63 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# NEVER globally replace this value.
#
# Validate the exact:
#
#     variable
#     count
#     CDC range/code
#
# before changing anything.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached WHQ_F.xpt:
#
#     WHD220 = 93 tiny values
#
#     TOTAL = 93
#
# CDC:
#
# WHD220 = Weight loss most successful (pounds)
#
# Published valid range:
#
#     0-248 = 3,849 observations
#
# After converting the 93 tiny values to zero,
# this attached XPT has exactly:
#
#     0-248 = 3,849
#     77777 = 1
#     99999 = 50
#     missing = 2,989
#
# This exactly matches CDC.
#

EXPECTED_TINY_COUNTS = {
    "WHD220": 93
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated WHQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 93:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Exact 93-value WHD220 "
    "tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED WHD220 ZEROS
# ============================================================

#
# Restore ONLY exact TINY_VALUE occurrences
# inside WHD220.
#
# No global replacement.
#

tiny_mask = (
    df["WHD220"]
    .eq(TINY_VALUE)
)


correction_count = int(
    tiny_mask.sum()
)


if correction_count != 93:

    raise ValueError(
        "Unexpected WHD220 tiny-value count."
    )


df.loc[
    tiny_mask,
    "WHD220"
] = 0


print("\n--- ZERO RESTORATION ---")


print(
    "WHD220 values restored to zero:",
    correction_count
)


if int(
    df["WHD220"]
    .eq(0)
    .sum()
) != 93:

    raise ValueError(
        "WHD220 zero restoration failed."
    )


print(
    "PASS: 93 validated WHD220 zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# dataset SHOULD contain.
#
# The ONLY numeric change expected from raw pandas XPT:
#
#     WHD220 tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WHD220"].eq(TINY_VALUE),
    "WHD220"
] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# NEVER round genuine decimals.
#
# Always run this test BEFORE Int64 conversion.
#
# Attached WHQ_F:
#
#     genuine fractional variables = NONE
#
# Although WHD fields include height/weight values,
# the PUBLIC RELEASE stores them here as whole-number
# inches/pounds.
#
# If any unexpected fraction appears, STOP rather
# than rounding it.
#
# This same rule protects genuine Dietary, Laboratory,
# Examination and sample-weight decimals.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional observations"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in WHQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT PROVEN WHOLE-NUMBER VALUES WITHOUT .0
# ============================================================

#
# After:
#
#     validated zero restoration
#     genuine-decimal check
#
# every WHQ_F variable is proven whole-number.
#
# Therefore all 63 variables can safely be
# converted to nullable Int64.
#
# IMPORTANT:
#
# NO .round() IS USED.
#

INTEGER_COLUMNS = df.columns.tolist()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 63 WHQ_F variables formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. HELPER - CATEGORICAL CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - CURRENT HEIGHT
# ============================================================

#
# WHD010:
#
# 41-81 = 6,730
# 7777  = 0
# 9999  = 104
# Missing = 55
#

s = df["WHD010"]


if not (
    int(
        s.between(
            41,
            81
        ).sum()
    )
    == 6730
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 104
    and
    int(
        s.isna()
        .sum()
    )
    == 55
):

    raise ValueError(
        "WHD010 differs from CDC."
    )


print(
    "PASS: WHD010 matches CDC."
)


# ============================================================
# 13. CDC CHECK - CURRENT WEIGHT
# ============================================================

#
# WHD020:
#
# 76-670 = 6,741
# 7777   = 6
# 9999   = 78
# Missing = 64
#

s = df["WHD020"]


if not (
    int(
        s.between(
            76,
            670
        ).sum()
    )
    == 6741
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 6
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 78
    and
    int(
        s.isna()
        .sum()
    )
    == 64
):

    raise ValueError(
        "WHD020 differs from CDC."
    )


print(
    "PASS: WHD020 matches CDC."
)


# ============================================================
# 14. CDC CHECK - WHQ030
# ============================================================

#
# How participant considers current weight.
#

check_frequency(
    "WHQ030",
    {
        1: 3535,
        2: 362,
        3: 2973,
        7: 1,
        9: 18
    },
    0
)


# ============================================================
# 15. CDC CHECK - WHQ040
# ============================================================

check_frequency(
    "WHQ040",
    {
        1: 578,
        2: 4099,
        3: 2205,
        7: 1,
        9: 6
    },
    0
)


# ============================================================
# 16. CDC CHECK - DESIRED WEIGHT
# ============================================================

#
# WHD045:
#
# 85-441 = 4,638
# 77777  = 1
# 99999  = 21
# Missing = 2,229
#

s = df["WHD045"]


if not (
    int(
        s.between(
            85,
            441
        ).sum()
    )
    == 4638
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 21
    and
    int(
        s.isna()
        .sum()
    )
    == 2229
):

    raise ValueError(
        "WHD045 differs from CDC."
    )


print(
    "PASS: WHD045 matches CDC."
)


# ============================================================
# 17. CDC CHECK - WEIGHT ONE YEAR AGO
# ============================================================

#
# WHD050:
#
# 80-586 = 6,707
# 7777   = 12
# 9999   = 118
# Missing = 52
#

s = df["WHD050"]


if not (
    int(
        s.between(
            80,
            586
        ).sum()
    )
    == 6707
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 12
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 118
    and
    int(
        s.isna()
        .sum()
    )
    == 52
):

    raise ValueError(
        "WHD050 differs from CDC."
    )


print(
    "PASS: WHD050 matches CDC."
)


# ============================================================
# 18. CDC CHECK - WEIGHT CHANGE INTENTIONAL
# ============================================================

check_frequency(
    "WHQ060",
    {
        1: 754,
        2: 497
    },
    5638
)


# ============================================================
# 19. CDC CHECK - TRIED TO LOSE WEIGHT
# ============================================================

check_frequency(
    "WHQ070",
    {
        1: 1916,
        2: 4203,
        7: 1,
        9: 9
    },
    760
)


# ============================================================
# 20. CDC CHECK - WEIGHT LOSS METHODS
# ============================================================

#
# These are CODE ALL THAT APPLY fields.
#
# Each populated variable contains its specific
# released method code.
#

WEIGHT_LOSS_METHODS = {

    "WHD080A": (10, 1538, 5351),
    "WHD080B": (11, 724, 6165),
    "WHD080C": (12, 839, 6050),
    "WHD080D": (13, 1604, 5285),
    "WHD080E": (14, 412, 6477),
    "WHD080F": (15, 247, 6642),
    "WHD080G": (16, 97, 6792),
    "WHD080H": (17, 146, 6743),

    "WHD080I": (31, 61, 6828),
    "WHD080J": (32, 130, 6759),
    "WHD080K": (33, 25, 6864),

    "WHD080L": (40, 28, 6861),
    "WHD080M": (34, 930, 5959),
    "WHD080N": (30, 106, 6783),

    "WHD080O": (41, 530, 6359),
    "WHD080P": (42, 24, 6865),
    "WHD080Q": (43, 977, 5912),
    "WHD080R": (44, 676, 6213),
    "WHD080S": (45, 736, 6153),
    "WHD080T": (46, 788, 6101)
}


print("\n--- CDC WEIGHT LOSS METHOD CHECKS ---")


for col, (
    expected_code,
    expected_count,
    expected_missing
) in WEIGHT_LOSS_METHODS.items():

    values = (
        df[col]
        .value_counts()
        .to_dict()
    )


    if values != {
        expected_code: expected_count
    }:

        raise ValueError(
            f"{col} differs from CDC.\n"
            f"Actual: {values}"
        )


    if int(
        df[col]
        .isna()
        .sum()
    ) != expected_missing:

        raise ValueError(
            f"{col}: missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 21. CDC CHECK - HEALTH PROFESSIONAL HELP
# ============================================================

check_frequency(
    "WHQ270",
    {
        1: 259,
        2: 2411
    },
    4219
)


# ============================================================
# 22. CDC CHECK - HEALTH PROFESSIONAL TYPES
# ============================================================

PROFESSIONAL_TYPES = {

    "WHQ280A": (1, 86, 6803),
    "WHQ280B": (2, 47, 6842),
    "WHQ280C": (3, 53, 6836),
    "WHQ280D": (4, 87, 6802),
    "WHQ280E": (5, 18, 6871)
}


print("\n--- CDC HEALTH PROFESSIONAL CHECKS ---")


for col, (
    expected_code,
    expected_count,
    expected_missing
) in PROFESSIONAL_TYPES.items():

    actual = (
        df[col]
        .value_counts()
        .to_dict()
    )


    if actual != {
        expected_code: expected_count
    }:

        raise ValueError(
            f"{col} differs from CDC."
        )


    if int(
        df[col]
        .isna()
        .sum()
    ) != expected_missing:

        raise ValueError(
            f"{col} missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 23. CDC CHECK - TRIED NOT TO GAIN WEIGHT
# ============================================================

check_frequency(
    "WHQ090",
    {
        1: 663,
        2: 3542,
        7: 1,
        9: 7
    },
    2676
)


# ============================================================
# 24. CDC CHECK - METHODS USED NOT TO GAIN WEIGHT
# ============================================================

WEIGHT_MAINTENANCE_METHODS = {

    "WHD100A": (10, 261, 6628),
    "WHD100B": (11, 94, 6795),
    "WHD100C": (12, 157, 6732),
    "WHD100D": (13, 365, 6524),
    "WHD100E": (14, 56, 6833),
    "WHD100F": (15, 20, 6869),
    "WHD100G": (16, 7, 6882),
    "WHD100H": (17, 6, 6883),

    "WHD100I": (31, 2, 6887),
    "WHD100J": (32, 10, 6879),
    "WHD100K": (33, 5, 6884),

    "WHD100L": (40, 6, 6883),
    "WHD100M": (34, 196, 6693),
    "WHD100N": (30, 1, 6888),

    "WHD100O": (41, 84, 6805),
    "WHD100P": (42, 3, 6886),
    "WHD100Q": (43, 219, 6670),
    "WHD100R": (44, 135, 6754),
    "WHD100S": (45, 148, 6741),
    "WHD100T": (46, 188, 6701)
}


print("\n--- CDC WEIGHT MAINTENANCE METHOD CHECKS ---")


for col, (
    expected_code,
    expected_count,
    expected_missing
) in WEIGHT_MAINTENANCE_METHODS.items():

    actual = (
        df[col]
        .value_counts()
        .to_dict()
    )


    if actual != {
        expected_code: expected_count
    }:

        raise ValueError(
            f"{col} differs from CDC."
        )


    if int(
        df[col]
        .isna()
        .sum()
    ) != expected_missing:

        raise ValueError(
            f"{col} missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 25. CDC CHECK - EVER TRIED TO LOSE WEIGHT
# ============================================================

check_frequency(
    "WHQ210",
    {
        1: 1257,
        2: 2950,
        7: 1,
        9: 5
    },
    2676
)


# ============================================================
# 26. CDC CHECK - WHD220
# ============================================================

#
# CRITICAL ZERO RESTORATION CHECK
#
# CDC:
#
# 0-248 = 3,849
# 77777 = 1
# 99999 = 50
# Missing = 2,989
#
# After correction:
#
# zero = 93
#

s = df["WHD220"]


print("\n--- WHD220 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-248:",
    int(
        s.between(
            0,
            248
        ).sum()
    )
)


print(
    "Refused:",
    int(
        s.eq(77777)
        .sum()
    )
)


print(
    "Don't know:",
    int(
        s.eq(99999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 93
    and
    int(
        s.between(
            0,
            248
        ).sum()
    )
    == 3849
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 50
    and
    int(
        s.isna()
        .sum()
    )
    == 2989
):

    raise ValueError(
        "WHD220 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: WHD220 matches CDC exactly."
)


# ============================================================
# 27. CDC CHECK - WEIGHT 10 YEARS AGO
# ============================================================

#
# WHD110:
#
# 80-450 = 4,327
# 7777   = 4
# 9999   = 187
# Missing = 2,371
#

s = df["WHD110"]


if not (
    int(
        s.between(
            80,
            450
        ).sum()
    )
    == 4327
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 4
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 187
    and
    int(
        s.isna()
        .sum()
    )
    == 2371
):

    raise ValueError(
        "WHD110 differs from CDC."
    )


print(
    "PASS: WHD110 matches CDC."
)


# ============================================================
# 28. CDC CHECK - WEIGHT AT AGE 25
# ============================================================

#
# WHD120:
#
# 50-441 = 5,107
# 7777   = 2
# 9999   = 298
# Missing = 1,482
#

s = df["WHD120"]


if not (
    int(
        s.between(
            50,
            441
        ).sum()
    )
    == 5107
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 2
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 298
    and
    int(
        s.isna()
        .sum()
    )
    == 1482
):

    raise ValueError(
        "WHD120 differs from CDC."
    )


print(
    "PASS: WHD120 matches CDC."
)


# ============================================================
# 29. CDC CHECK - HEIGHT AT AGE 25
# ============================================================

#
# WHD130:
#
# 41-80 = 2,873
# 7777  = 0
# 9999  = 142
# Missing = 3,874
#

s = df["WHD130"]


if not (
    int(
        s.between(
            41,
            80
        ).sum()
    )
    == 2873
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 142
    and
    int(
        s.isna()
        .sum()
    )
    == 3874
):

    raise ValueError(
        "WHD130 differs from CDC."
    )


print(
    "PASS: WHD130 matches CDC."
)


# ============================================================
# 30. CDC CHECK - GREATEST WEIGHT
# ============================================================

#
# WHD140:
#
# 86-717 = 6,399
# 7777   = 8
# 9999   = 98
# Missing = 384
#

s = df["WHD140"]


if not (
    int(
        s.between(
            86,
            717
        ).sum()
    )
    == 6399
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 8
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 98
    and
    int(
        s.isna()
        .sum()
    )
    == 384
):

    raise ValueError(
        "WHD140 differs from CDC."
    )


print(
    "PASS: WHD140 matches CDC."
)


# ============================================================
# 31. CDC CHECK - AGE WHEN HEAVIEST
# ============================================================

#
# WHQ150:
#
# 7-79 = 6,236
# 80   = 122
# 77777 = 1
# 99999 = 38
# Missing = 492
#

s = df["WHQ150"]


if not (
    int(
        s.between(
            7,
            79
        ).sum()
    )
    == 6236
    and
    int(
        s.eq(80)
        .sum()
    )
    == 122
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 38
    and
    int(
        s.isna()
        .sum()
    )
    == 492
):

    raise ValueError(
        "WHQ150 differs from CDC."
    )


print(
    "PASS: WHQ150 matches CDC."
)


# ============================================================
# 32. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected numeric difference from raw pandas XPT:
#
# WHD220:
#
#     93 TINY_VALUE values -> 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 63 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 33. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 34. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 35. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 63 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 36. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 37. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 38. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# WHQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere in
# this particular CSV.
#
# IMPORTANT:
#
# DO NOT use this global test on Dietary or any
# mixed-decimal Lab/Examination dataset.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 39. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 40. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 41. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "WHQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "WHD220 zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 WHQ_F XPT -> CSV
Rows: 6,889
Columns: 63
PASS: Structure = 6,889 rows x 63 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 93

Tiny values by variable:
WHD220    93

PASS: Exact 93-value WHD220 tiny-zero pattern validated.

--- ZERO RESTORATION ---
WHD220 values restored to zero: 93
PASS: 93 validated WHD220 zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in WHQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All 63 WHQ_F variables formatted without unnecessary .0.
PASS: WHD010 matches CDC.
PASS: WHD020 matches CDC.
PASS: WHQ030 matches CDC.
PASS: WHQ040 matches CDC.
PASS: WHD045 matches CDC.
PASS: WHD050 matches CDC.
PASS: WHQ060 matches CDC.
PASS: WHQ070 matches CDC.

--- CDC WEIGHT LOSS METHOD CHECKS ---
PASS: WHD080A matches CDC.
PASS: WHD080B matches CDC.
PASS: WHD080C matches CDC.
PASS: WHD080D matches CDC.
PASS: WHD080E matches